In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path(r"D:\shrey\Documents\ML- PROJECT")
sys.path.insert(0, str(ROOT_DIR))
import json, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import config as cfg
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate

df = pd.read_csv(cfg.PROCESSED_DIR / "placement_clean.csv")
for c in ["eligible", "oa_cleared", "selected"]:
    df[c] = df[c].astype(str) == "True"          # NaN becomes False; only used inside stage subsets

eligible_df = df[df["eligible"]].copy()          # took the OA (2,990)
d = df[df["oa_cleared"]].copy()                  # reached interviews (960)
d["selected"] = d["selected"].astype(int)

ACAD = ["tenth_pct", "twelfth_pct", "jee_percentile", "cgpa", "jee_missing"]
OA = list(cfg.OA_SECTIONS)
print(len(eligible_df), len(d), d["selected"].sum())   # 2990 960 96

2990 960 96


In [2]:
h = d.dropna(subset=["technical", "oa_total", "coding"]).reset_index(drop=True)
Xa = np.column_stack([np.ones(len(h)), h["oa_total"], h["coding"]])
ya = h["technical"].to_numpy()

beta = np.linalg.lstsq(Xa, ya, rcond=None)[0]
print(f"oa_total: {beta[1]:.3f} (planted 0.65 + indirect) | coding: {beta[2]:.3f} (planted {cfg.TECHNICAL['direct_coding_boost_on_technical']})")

rng = np.random.default_rng(cfg.SEED)
by_student = h.groupby("student_id").indices
keys = list(by_student)
boots = []
for _ in range(1000):
    pick = rng.integers(0, len(keys), len(keys))
    rows = np.concatenate([by_student[keys[i]] for i in pick])
    boots.append(np.linalg.lstsq(Xa[rows], ya[rows], rcond=None)[0][2])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"coding coefficient 95% bootstrap interval: [{lo:.3f}, {hi:.3f}]")

oa_total: 0.669 (planted 0.65 + indirect) | coding: 0.152 (planted 0.25)
coding coefficient 95% bootstrap interval: [0.069, 0.237]


In [3]:
Xh, yh, gh = h[["oa_total", "coding"]], h["technical"], h["student_id"]
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=cfg.SEED).split(Xh, yh, gh))
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=cfg.SEED).fit(Xh.iloc[tr], yh.iloc[tr])
print("held-out R2:", round(rf.score(Xh.iloc[te], yh.iloc[te]), 3))

pi = permutation_importance(rf, Xh.iloc[te], yh.iloc[te], n_repeats=30, random_state=cfg.SEED, scoring="r2")
print(pd.DataFrame({"mean_drop_in_R2": pi.importances_mean, "sd": pi.importances_std}, index=Xh.columns).round(4))

held-out R2: 0.147
          mean_drop_in_R2      sd
oa_total           0.3144  0.0572
coding             0.0593  0.0226


In [4]:
def reg_pipe(kind):
    if kind == "dummy":
        return DummyRegressor(strategy="mean")
    steps = [("imp", SimpleImputer(strategy="median"))]
    if kind == "linear":
        steps += [("sc", StandardScaler()), ("m", LinearRegression())]
    else:
        steps += [("m", RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=cfg.SEED))]
    return Pipeline(steps)

scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}

def eval_reg(data, feats, target, label):
    rows = []
    for kind in ["dummy", "linear", "rf"]:
        r = cross_validate(reg_pipe(kind), data[feats], data[target], groups=data["student_id"],
                           cv=GroupKFold(n_splits=5), scoring=scoring)
        rows.append({"task": label, "model": kind,
                     "MAE": -r["test_mae"].mean(), "RMSE": -r["test_rmse"].mean(), "R2": r["test_r2"].mean()})
    return pd.DataFrame(rows).round(3)

reg_final = eval_reg(d, ACAD + OA, "final_score", "final_score from academics + OA")
print(reg_final)

                              task   model    MAE   RMSE     R2
0  final_score from academics + OA   dummy  5.162  6.376 -0.007
1  final_score from academics + OA  linear  3.814  4.711  0.448
2  final_score from academics + OA      rf  4.021  4.985  0.383


In [5]:
reg_oa = eval_reg(eligible_df.dropna(subset=["oa_total"]), ACAD, "oa_total", "oa_total from academics")
print(reg_oa)

reg_results = pd.concat([reg_oa, reg_final])
reg_results.to_csv(cfg.PROCESSED_DIR / "regression_results.csv", index=False)

                      task   model    MAE   RMSE     R2
0  oa_total from academics   dummy  7.344  9.107 -0.001
1  oa_total from academics  linear  4.360  5.448  0.641
2  oa_total from academics      rf  4.542  5.656  0.613


In [6]:
w = d.dropna(subset=["oa_total", "technical", "hr", "final_score"])
m = LinearRegression().fit(w[["oa_total", "technical", "hr"]], w["final_score"])
check = pd.DataFrame({"planted": [cfg.FINAL_SCORE_WEIGHTS["oa"], cfg.FINAL_SCORE_WEIGHTS["technical"], cfg.FINAL_SCORE_WEIGHTS["hr"]],
                      "recovered": m.coef_.round(3)}, index=["oa_total", "technical", "hr"])
print(check, "| intercept:", round(m.intercept_, 3))

           planted  recovered
oa_total      0.40      0.401
technical     0.35      0.349
hr            0.25      0.249 | intercept: 0.01


In [7]:
cfg.MODELS_DIR.mkdir(exist_ok=True)

# 1) Stage 1: expected OA score from academics
m1 = reg_pipe("linear").fit(eligible_df.dropna(subset=["oa_total"])[ACAD], eligible_df.dropna(subset=["oa_total"])["oa_total"])

# 2) Stage 2: probability of final selection after the OA (Model B, logistic regression)
clf = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))])
m2 = clf.fit(d[ACAD + OA], d["selected"])

# 3) Expected final score after the OA
m3 = reg_pipe("linear").fit(d[ACAD + OA], d["final_score"])

joblib.dump(m1, cfg.MODELS_DIR / "stage1_oa_regressor.joblib")
joblib.dump(m2, cfg.MODELS_DIR / "stage2_selection_classifier.joblib")
joblib.dump(m3, cfg.MODELS_DIR / "stage2_final_score_regressor.joblib")
json.dump({"stage1": ACAD, "stage2": ACAD + OA}, open(cfg.MODELS_DIR / "feature_lists.json", "w"), indent=2)
print("saved:", sorted(p.name for p in cfg.MODELS_DIR.iterdir()))

saved: ['feature_lists.json', 'stage1_oa_regressor.joblib', 'stage2_final_score_regressor.joblib', 'stage2_selection_classifier.joblib']
